# Applying interventions to the SIR model
## Basic model
### Model construction
The model in this notebook is similar to the one presented in `sir-teaching.ipynb`, but adapted so that we can implement some interventions.

In [ ]:
%pip install summerepi2==1.3.6

The next cell imports some standard libraries
You don't need to worry about what these are,
although if you have a background in data science
some of them will probably be familiar.
There are some comments to explain what these are
in case you are interested to learn more.

In [ ]:
import pandas as pd  # Pandas for data wrangling
import plotly.express as px  # Plotly for interactive visualisation
import plotly.graph_objects as go
from plotly.subplots import make_subplots
pd.options.plotting.backend = "plotly"

from summer2 import CompartmentalModel  # Our "summer" platform for building epidemic models
from summer2.parameters import Parameter, DerivedOutput
from summer2.functions.time import get_linear_interpolation_function as linear_interp

### Building the model
This time, let's build a slightly more complicated model so that we can distinguish two phases

In [ ]:
total_population = 7e6
infectious_seed = 1.0
run_period = [0.0, 50.0]
model_comps = ["vaccinated", "susceptible", "infectious_asympt", "infectious_sympt", "recovered"]
infect_comps = ["infectious_asympt", "infectious_sympt"]
sir_model = CompartmentalModel(times=run_period, compartments=model_comps, infectious_compartments=infect_comps)

In [ ]:
infection_rate = Parameter("contact_rate") * (1.0 - Parameter("face_mask_coverage") * Parameter("face_mask_efficacy"))
sir_model.add_infection_frequency_flow(name="infection", contact_rate=infection_rate, source="susceptible", dest="infectious_asympt")

In [ ]:
vacc_infection_rate = infection_rate * Parameter("vacc_efficacy")
sir_model.add_infection_frequency_flow(name="infection_vacc", contact_rate=infection_rate, source="vaccinated", dest="infectious_asympt")

In [ ]:
sir_model.add_transition_flow(name="progression", fractional_rate=Parameter("recovery_rate") * 2.0, source="infectious_asympt", dest="infectious_sympt")
resolve_sympt_rate = Parameter("isolation_rate") + Parameter("recovery_rate") * 2.0
sir_model.add_transition_flow(name="recovery", fractional_rate=resolve_sympt_rate, source="infectious_sympt", dest="recovered")

### Running the model

In [ ]:
start_pop = {"susceptible": total_population - infectious_seed, "infectious_asympt": infectious_seed}
sir_model.set_initial_population(start_pop)
parameters = {
    "contact_rate": 1.5,
    "recovery_rate": 0.2,
    "face_mask_coverage": 0.0,
    "face_mask_efficacy": 0.2,
    "isolation_rate": 1.0,
    "vacc_efficacy": 1.0,
}
sir_model.run(parameters)
sir_model.get_outputs_df().plot(labels={"index": "time", "value": "number of people"})